# Lab | Summarization evaluation using LangSmith
Let's revisit your capstone project 2? Well, sort of. Pick diffierent sets of data and re-run this notebook. Maybe parts of the dataset you used in your last project week. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications using LangSmith.

What did you learn? - Let's discuss that in class

## LangSmith - LangChain evaluation

> ⚠️ **Updated:** Original notebook loaded API keys from a `.env` file using `dotenv`. This updated version uses `getpass` so you can enter keys directly at runtime without needing a `.env` file.

In [1]:
import os
from getpass import getpass

OPENAI_API_KEY = getpass('Please enter your OpenAI API Key: ')
LANGCHAIN_API_KEY = getpass('Please enter your LangChain API Key: ')
HUGGINGFACEHUB_API_TOKEN = getpass('Please enter your HuggingFace Hub API Token: ')

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['LANGCHAIN_API_KEY'] = LANGCHAIN_API_KEY
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HUGGINGFACEHUB_API_TOKEN

Please enter your OpenAI API Key: ··········
Please enter your LangChain API Key: ··········
Please enter your HuggingFace Hub API Token: ··········


In [2]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]="langsmith_max-test"

In [3]:
#Importing Client from Langsmith
from langsmith import Client
client = Client(api_key=LANGCHAIN_API_KEY)

### Create Dataset


> ⚠️ **Updated:** Original used `ccdv/cnn_dailymail` with `version=` and `trust_remote_code=True` which are no longer supported. Updated to use `cnn_dailymail` with `name=` parameter.

In [4]:
from datasets import load_dataset
cnn_dataset = load_dataset("cnn_dailymail", name="3.0.0")

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [5]:
def add_prefix(example):
    return {
        **example,
        "article": f"Summarize this news:\n{example['article']}"
    }

#cnn_dataset = cnn_dataset.map(add_prefix)

In [6]:
cnn_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [7]:
cnn_dataset['train'][0]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [8]:
#Get just a few news to test
MAX_NEWS=10
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS)).map(add_prefix)

sample_cnn

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 10
})

The dataset contains three columns: article, highlights, and id. To use LangSmith, we need to create a dataset in LangSmith format.

LangSmith expects a prompt and a result. To achieve this, we will transform the article into a prompt by adding the prefix: "Summarize this news." As a result, we will use the content of highlights, which represents the summaries created by humans.

In [9]:
print(sample_cnn[0])

{'article': 'Summarize this news:\n(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki,

Now We have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith with this information.
### Create the Dataset in Langsmith

The dataset in LangSmith is composed of an input, which is the prompt passed to the model for evaluation, and an output, which should contain what we expect the model to return.

In [10]:
import datetime

In [11]:
import uuid
input_key=['article']
output_key=['highlights']

NAME_DATASET=f"Summarize_dataset_{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

In [12]:
#This creates the dataset in LangSmith with the content in sample_cnn - If you run this more than once you will get POST errors
dataset = client.upload_dataframe(
    df=sample_cnn,
    input_keys=input_key,
    output_keys=output_key,
    name=NAME_DATASET,
    description="Test Embedding distance between model summarizations",
    data_type="kv"
)
print(f"✅ Dataset created: {NAME_DATASET}")

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅ Dataset created: Summarize_dataset_2026-03-05 16:53:23


In this image, we can see an example from the dataset once it's been registered in LangSmith.

In the Input column, there is the prompt to be sent, while in the Output column, the expected output is stored.

When performing the comparison, the model will be given the prompt, and the Cosine distance between its response and the one stored in the sample dataset will be calculated.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Dataset.jpg?raw=true">

### Recovering Models From Hugging Face
Let's retrieve both models from HuggingFace. A base model and a model that has been fine-tuned using the training portion of this same dataset to generate summaries.

> ⚠️ **Updated:** Original used `HuggingFaceHub` from `langchain` which is deprecated, and `t5-base` / `flax-community/t5-base-cnn-dm` which are no longer accessible via the HuggingFace Inference API. Updated to use direct HTTP requests to the HuggingFace Router API with `facebook/bart-large-cnn` as the base model (a reliable summarization model) and kept `flax-community/t5-base-cnn-dm` as the fine-tuned model.

In [13]:
import requests
import numpy as np
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)

def hf_summarize(text, repo_id):
    API_URL = f'https://router.huggingface.co/hf-inference/models/{repo_id}'
    headers = {'Authorization': f'Bearer {HUGGINGFACEHUB_API_TOKEN}'}
    try:
        response = requests.post(API_URL, headers=headers, json={'inputs': text[:1024]})
        if response.status_code == 200:
            result = response.json()
            if isinstance(result, list):
                return result[0].get('summary_text') or result[0].get('generated_text', '')
            return str(result)
        return f'API Error: {response.status_code}'
    except Exception as e:
        return f'Error: {str(e)}'

# Base model: facebook/bart-large-cnn (replaces t5-base)
def summarizer_base(inputs):
    return hf_summarize(inputs, 'facebook/bart-large-cnn')

# Fine-tuned model: flax-community/t5-base-cnn-dm
def summarizer_finetuned(inputs):
    return hf_summarize(inputs, 'flax-community/t5-base-cnn-dm')

## Defining Evaluator
The first step is to define an evaluator, where we specify the variables we want to evaluate. In our case, I have chosen to measure only the "embedding_distance."

I've left the "string_distance" as a comment in case you want to conduct a test with two evaluations instead of one.

> ⚠️ **Updated:** Original used `run_on_dataset` and `RunEvalConfig` from `langchain.smith` which have been removed in newer LangChain versions. Updated to use `langsmith.evaluate` with a custom `embedding_distance_evaluator` function that computes cosine distance using OpenAI embeddings.

In [14]:
from langsmith import evaluate

def get_embedding(text):
    if not text or not str(text).strip():
        return np.zeros(1536)
    response = openai_client.embeddings.create(
        input=str(text)[:8000],
        model='text-embedding-3-small'
    )
    return np.array(response.data[0].embedding)

def embedding_distance_evaluator(run, example):
    prediction = (run.outputs or {}).get('output', '')
    reference = (example.outputs or {}).get('highlights', '')
    emb1 = get_embedding(str(prediction))
    emb2 = get_embedding(str(reference))
    cosine_sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2) + 1e-9)
    return {'key': 'embedding_distance', 'score': 1 - cosine_sim}

#We are using just one of the multiple evaluators available on LangSmith.
#string_distance is left as a comment in case you want to add a second evaluator.
#string_distance_evaluator can be added to the evaluators list similarly.

### Running Evaluator
With the same configuration, we can launch two evaluations on the same dataset. One for each of the chosen models.

> ⚠️ **Updated:** Original used `run_on_dataset` with `summarizer_base` (t5-base). Updated to use `langsmith.evaluate` with `facebook/bart-large-cnn` as the base model since t5-base is no longer accessible via the HuggingFace Inference API.

In [15]:
def summarize_bart(inputs: dict):
    output = hf_summarize(inputs['article'], 'facebook/bart-large-cnn')
    return {'output': output}

bart_results = evaluate(
    summarize_bart,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix='BART-large-CNN',
    client=client,
)

View the evaluation results for experiment: 'BART-large-CNN-fe31fd43' at:
https://smith.langchain.com/o/aac18a1f-a6be-49d2-b7b6-9821182bad4b/datasets/7e80e98a-b615-4cae-9b0a-757b8127409e/compare?selectedSessions=6b5f269f-53e2-4baa-83a8-542a4a31cf75




0it [00:00, ?it/s]

> ⚠️ **Updated:** Original used `run_on_dataset` with `summarizer_finetuned`. Updated to use `langsmith.evaluate` with a direct HuggingFace API call.

In [16]:
def summarize_t5_finetuned(inputs: dict):
    output = hf_summarize(inputs['article'], 'flax-community/t5-base-cnn-dm')
    return {'output': output}

finetuned_t5_results = evaluate(
    summarize_t5_finetuned,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix='T5-FineTuned',
    client=client,
)

View the evaluation results for experiment: 'T5-FineTuned-926a8155' at:
https://smith.langchain.com/o/aac18a1f-a6be-49d2-b7b6-9821182bad4b/datasets/7e80e98a-b615-4cae-9b0a-757b8127409e/compare?selectedSessions=bf85f3fe-e286-48ca-98b3-ab5b1747e254




0it [00:00, ?it/s]

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Tests.jpg?raw=true">

In the image below you can see the comparision between two tests.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareTestst.jpg?raw=true">

Well, since it has been so straightforward, why don't we try to make the comparison with an OpenAI model?

> ⚠️ **Updated:** Original used `from langchain.chat_models import ChatOpenAI` which is deprecated, and `run_on_dataset` which has been removed. Updated to use `from langchain_openai import ChatOpenAI` and `langsmith.evaluate`.

In [17]:
import sys
!{sys.executable} -m pip install langchain-openai

from langchain_openai import ChatOpenAI

open_aillm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.0, openai_api_key=OPENAI_API_KEY)

def summarize_openai(inputs: dict):
    article = inputs['article'][:3000]
    result = open_aillm.invoke(article)
    return {'output': result.content if result.content else 'No summary generated'}

openai_results = evaluate(
    summarize_openai,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix='OpenAI-GPT35',
    client=client,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.6 MB/s eta 0:00:00
View the evaluation results for experiment: 'OpenAI-GPT35-5a02710e' at:
https://smith.langchain.com/o/aac18a1f-a6be-49d2-b7b6-9821182bad4b/datasets/7e80e98a-b615-4cae-9b0a-757b8127409e/compare?selectedSessions=bc62cadc-da49-40d7-aa98-9b6999850137




0it [00:00, ?it/s]

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareOpenAI_HF.jpg?raw=true">

The experiment with the OpenAI model has yielded the best results. But, be aware! As we can see, there is a cost involved since we are using an API, and it needs to be paid for.

Another crucial piece of information is that we can view performance data for the models. This data could also be useful for minimally evaluating our inference server.